# Benchmark 

### Bloque 1: Importación de librerías y conecxión

In [ ]:
# Importamos lo necesario. time para medir y pandas para las tablas del paper
import time
import pandas as pd
from graphdatascience import GraphDataScience

# Nos conectamos a Neo4j. 
# Ojo: cambiar usuario y contraseña según lo que pusimos en el docker-compose
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "fraudelink123")

gds = GraphDataScience(URI, auth=AUTH)

# Lista para guardar los tiempos y luego armar el DataFrame
resultados_bench = []

print("Conexión exitosa al servidor GDS")

### Bloque 2: Proyección del Grafo

In [ ]:
print("Iniciando proyección del grafo en memoria...")
start_time = time.time()

# SOLAMENTE proyectamos Cuentas y TRANSFERENCIAS para aislar la red de fraude
G, project_result = gds.graph.project(
    "grafo_fraude",
    "Cuenta",
    {"TRANSFIERE_A": {"orientation": "UNDIRECTED"}}
)

tiempo_proyeccion = time.time() - start_time
resultados_bench.append({"Prueba": "1. Proyección del Grafo", "Tiempo (s)": round(tiempo_proyeccion, 4)})

print(f"Grafo proyectado con {G.node_count()} nodos y {G.relationship_count()} relaciones.")
print(f"Duración: {tiempo_proyeccion:.4f} segundos.")

### Bloque 3: Benchmarking WCC (Componentes fuertemente conexos)

In [ ]:
print("\nEjecutando WCC...")
start_time = time.time()
# Escribe la propiedad 'wcc_comunidad' directamente en los nodos de Neo4j
wcc_result = gds.wcc.write(G, writeProperty="wcc_comunidad")
tiempo_wcc = time.time() - start_time
resultados_bench.append({"Prueba": "WCC (Write DB)", "Tiempo (s)": round(tiempo_wcc, 4)})
print(f"WCC listo. Escribió en la BD en {wcc_result['writeMillis']} ms.")

### Bloque 4: Celda 4: Benchmarking Louvain (Para encontrar redes más complejas de fraude)

In [ ]:
print("\nEjecutando algoritmo de Louvain...")
start_time = time.time()

# Escribe la propiedad 'louvain_comunidad' directamente en los nodos de Neo4j
louvain_result = gds.louvain.write(G, writeProperty="louvain_comunidad")
tiempo_louvain = time.time() - start_time
resultados_bench.append({"Prueba": "3. Louvain (Write DB)", "Tiempo (s)": round(tiempo_louvain, 4)})
print(f"Louvain listo. Escribió en la BD en {louvain_result['writeMillis']} ms.")

### Bloque 5: Limpieza de memoria

In [ ]:
# Borramos el grafo de la memoria RAM de Neo4j para no botar el server
G.drop()
print("Grafo eliminado de la memoria GDS exitosamente.")

### Bloque 6: Resultados y exportación para reporte

In [ ]:
# Armamos la tabla con los resultados
df_metricas = pd.DataFrame(resultados_bench)

print("\n--- Métricas de Desempeño (Requisito 5) ---")
display(df_metricas) # display() hace que la tabla se vea bonita en Jupyter

# Exportamos a CSV por si ocupamos los datos crudos para el reporte
df_metricas.to_csv("metricas_rendimiento_gds.csv", index=False)
print("Tabla guardada como metricas_rendimiento_gds.csv")